In [ ]:
'''Quick revision: Which asyncio method should you use?
Method	                         Main purpose
asyncio.create_task()	Schedule a coroutine to run concurrently
asyncio.gather()	    Run multiple awaitables and collect results
asyncio.wait()	        Wait for tasks based on completion conditions or timeout
asyncio.as_completed()	Process results in completion order
asyncio.wait_for()	    Enforce a timeout on one awaitable
task.cancel()	        Request task cancellation
add_done_callback()	    Run a callback when a task completes
asyncio.Lock()	        Protect shared state from concurrent access
asyncio.Queue()	        Safely pass work between async producers and consumers

The most important distinction to remember is that asyncio concurrency is particularly useful for I/O-bound work, such as API requests, file downloads, and database queries. It doesn't automatically make CPU-intensive calculations run in parallel.'''

In [ ]:
'''1.	Task Scheduling Problem
Write an asyncio program where two API calls take different times to complete. Use asyncio.create_task() so both start immediately, and print which task finishes first.'''


In [ ]:
import asyncio
import time

async def api_call(name, delay):
    print(f"{name} started")
    await asyncio.sleep(delay)
    print(f"{name} finished")
    return name

async def main():
    start = time.perf_counter()

    task1 = asyncio.create_task(api_call("API 1", 3))
    task2 = asyncio.create_task(api_call("API 2", 1))

    done, pending = await asyncio.wait(
        [task1, task2],
        return_when=asyncio.FIRST_COMPLETED
    )

    first_task = next(iter(done))
    print(f"First task to finish: {first_task.result()}")

    await asyncio.gather(*pending)
    print(f"Total time: {time.perf_counter() - start:.2f} seconds")

asyncio.run(main())

In [ ]:
'''Explanation:
- asyncio.create_task() schedules both coroutines to run concurrently.
- API 1 takes 3 seconds; API 2 takes 1 second.
- asyncio.wait(..., return_when=asyncio.FIRST_COMPLETED) returns as soon as at least one task finishes.
- We then await any remaining tasks so they finish cleanly.
Expected first completion: API 2.'''

In [ ]:
'''2.	Sequential vs Concurrent Execution Problem
Create three coroutines that each take 2 seconds. First run them using normal await, then run them using asyncio.gather(). Compare total execution time.'''

In [ ]:
import asyncio
import time

async def work(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")

async def main():
    start = time.perf_counter()

    print("Sequential execution")
    await work("Task 1")
    await work("Task 2")
    await work("Task 3")

    print(f"Sequential time: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()

    print("\nConcurrent execution")
    await asyncio.gather(
        work("Task 1"),
        work("Task 2"),
        work("Task 3")
    )

    print(f"Concurrent time: {time.perf_counter() - start:.2f}s")

asyncio.run(main())

In [ ]:
'''3.	Gather with Errors Problem
Build three async tasks: two return data successfully and one raises an exception. Use asyncio.gather(..., return_exceptions=True) to collect all results without stopping the program.'''


In [ ]:
import asyncio

async def fetch_data(name, fail=False):
    await asyncio.sleep(1)

    if fail:
        raise ValueError(f"{name} failed")

    return f"{name} data received"

async def main():
    results = await asyncio.gather(
        fetch_data("API 1"),
        fetch_data("API 2", fail=True),
        fetch_data("API 3"),
        return_exceptions=True
    )

    for result in results:
        if isinstance(result, Exception):
            print("Error:", result)
        else:
            print("Success:", result)

asyncio.run(main())

In [ ]:
'''4.	Wait with Timeout Problem
Create three tasks simulating file downloads: 1 sec, 3 sec, and 5 sec. Use asyncio.wait() with a timeout of 2 seconds. Print completed downloads and cancel pending ones.'''

In [ ]:
import asyncio

async def download(name, delay):
    try:
        print(f"{name} started")
        await asyncio.sleep(delay)
        print(f"{name} completed")
        return name
    except asyncio.CancelledError:
        print(f"{name} cancelled")
        raise

async def main():
    tasks = [
        asyncio.create_task(download("File 1", 1)),
        asyncio.create_task(download("File 2", 3)),
        asyncio.create_task(download("File 3", 5))
    ]

    done, pending = await asyncio.wait(
        tasks,
        timeout=2
    )

    print("\nCompleted downloads:")
    for task in done:
        print(task.result())

    print("\nCancelling pending downloads:")
    for task in pending:
        task.cancel()

    await asyncio.gather(*pending, return_exceptions=True)

asyncio.run(main())

In [ ]:
'''Explanation:
- done contains tasks completed within the timeout.
- pending contains tasks that are still running after 2 seconds.
- task.cancel() requests cancellation.
- await asyncio.gather(*pending, return_exceptions=True) lets the cancellations finish cleanly.
Expected outcome: File 1 completes, while Files 2 and 3 are cancelled.'''

In [ ]:
'''5.	Process Results as They Complete Problem
Simulate three web requests with different delays. Use asyncio.as_completed() to process each response immediately when it finishes instead of waiting for all.'''


In [ ]:
import asyncio

async def web_request(name, delay):
    await asyncio.sleep(delay)
    return f"{name} response received after {delay}s"

async def main():
    requests = [
        web_request("Website A", 3),
        web_request("Website B", 1),
        web_request("Website C", 2)
    ]

    for completed in asyncio.as_completed(requests):
        result = await completed
        print(result)

asyncio.run(main())

In [ ]:
'''expected
Website B response received after 1s
Website C response received after 2s
Website A response received after 3s

Remember the difference:
- asyncio.gather() returns results in input order.
- asyncio.as_completed() lets you process results in completion order.
This is useful when fetching data from several APIs and displaying each response as soon as it arrives.'''

In [ ]:
'''6.	Timeout Protection Problem
Write a coroutine that simulates a slow database query. Use asyncio.wait_for() to cancel it if it takes more than 2 seconds, and print "Database timeout".'''

In [ ]:
import asyncio

async def database_query():
    print("Database query started")
    await asyncio.sleep(5)
    return "Database result"

async def main():
    try:
        result = await asyncio.wait_for(
            database_query(),
            timeout=2
        )
        print(result)

    except asyncio.TimeoutError:
        print("Database timeout")

asyncio.run(main())

In [ ]:
'''Expecyed
Database query started
Database timeout


Key concept: asyncio.wait_for() applies a timeout to one awaitable. When the timeout expires, it requests cancellation of the operation and raises asyncio.TimeoutError (which is an alias of TimeoutError in modern Python).

Unlike asyncio.wait(), which returns pending tasks after a timeout, wait_for() cancels the timed-out operation by default.'''

In [ ]:
'''7.	Cancellation Problem
Create a long-running background task like sending email notifications. Cancel it after 2 seconds and handle asyncio.CancelledError properly.'''

In [ ]:
import asyncio

async def send_notifications():
    try:
        while True:
            print("Sending email notification...")
            await asyncio.sleep(1)

    except asyncio.CancelledError:
        print("Email notifications cancelled")
        raise

async def main():
    task = asyncio.create_task(send_notifications())

    await asyncio.sleep(2)

    print("Requesting cancellation...")
    task.cancel()

    try:
        await task
    except asyncio.CancelledError:
        print("Background task stopped")

asyncio.run(main())

In [ ]:
'''Execution flow:
1. The notification task starts in the background.
2. The main coroutine waits for 2 seconds.
3. task.cancel() requests cancellation.
4. The notification coroutine handles CancelledError.
5. The main coroutine awaits the task and handles the cancellation too.
Important: Cancellation is cooperative. An async coroutine normally receives the cancellation request when it reaches an appropriate suspension point, such as await asyncio.sleep().'''

In [ ]:
'''8.	Callback Problem
Create an async task that returns a calculated value. Attach a callback using add_done_callback() to print the result as soon as the task is completed.'''

In [ ]:
import asyncio

async def calculate():
    await asyncio.sleep(2)
    return 10 + 20

def show_result(task):
    if task.cancelled():
        print("Task was cancelled")
        return

    error = task.exception()
    if error is not None:
        print("Task failed:", error)
    else:
        print("Calculated result:", task.result())

async def main():
    task = asyncio.create_task(calculate())

    task.add_done_callback(show_result)

    print("Calculation started")

    await task

asyncio.run(main())

In [ ]:
'''- add_done_callback() registers a regular callback function.
- The callback receives the completed Task object.
- task.result() retrieves the return value.
- task.exception() checks whether the task raised an exception.
The callback runs when the task finishes. If you also await the task, as shown here, the program stays alive until the calculation completes.'''

In [ ]:
'''9.	Lock Synchronization Problem
Create three coroutines trying to update the same shared account balance. Use asyncio.Lock() so only one coroutine updates the balance at a time.'''


In [ ]:
import asyncio

balance = 1000

async def update_balance(name, amount, lock):
    global balance

    async with lock:
        print(f"{name} acquired the lock")

        old_balance = balance
        await asyncio.sleep(1)

        balance = old_balance + amount

        print(f"{name} updated balance to {balance}")

async def main():
    lock = asyncio.Lock()

    await asyncio.gather(
        update_balance("User 1", 100, lock),
        update_balance("User 2", 200, lock),
        update_balance("User 3", -50, lock)
    )

    print("Final balance:", balance)

asyncio.run(main())

In [ ]:
'''exp:
Final balance: 1250

Why is the lock necessary?

Imagine all three coroutines read the original balance of 1000 before any of them writes the updated balance. They could overwrite one another's updates, causing a lost-update bug.

async with lock: ensures that only one coroutine at a time executes the protected section. The lock is released automatically when that section exits.'''

In [ ]:
'''10.	Producer–Consumer Queue Problem
Build an async producer that adds 10 jobs into an asyncio.Queue, and two consumers that process the jobs concurrently. Use queue.join() and cancel consumers after all jobs are done.'''

In [ ]:
import asyncio

async def producer(queue):
    for job in range(1, 11):
        await queue.put(job)
        print(f"Produced job {job}")

    # Signal that no more jobs will be produced.
    await queue.put(None)
    await queue.put(None)

async def consumer(name, queue):
    while True:
        job = await queue.get()

        try:
            if job is None:
                print(f"{name} stopping")
                return

            print(f"{name} processing job {job}")
            await asyncio.sleep(1)
            print(f"{name} completed job {job}")

        finally:
            queue.task_done()

async def main():
    queue = asyncio.Queue()

    consumers = [
        asyncio.create_task(consumer("Consumer 1", queue)),
        asyncio.create_task(consumer("Consumer 2", queue))
    ]

    await producer(queue)

    # Wait until all jobs, including stop signals, are acknowledged.
    await queue.join()

    # Consumers have received their stop signals and can finish.
    await asyncio.gather(*consumers)

    print("All jobs completed")

asyncio.run(main())